# Step 4 — BPD Energy Benchmarks by Climate Zone

This notebook turns **Building Performance Database (BPD)** downloads into one benchmark table: for each ASHRAE
climate zone and school building type, the distribution (count, mean, percentiles) of **Energy Use Intensity (EUI)**
— annual energy use per square foot, in kBtu/ft²·year.

Benchmarks are taken **by climate zone, not by state**: within one state there are too few school buildings in BPD
for a stable comparison.

**Input:** BPD table downloads, one CSV per climate zone and metric (section 1).
**Output:** `data/interim/bpd_eui_benchmarks.csv` — one row per climate zone × building type, with Site, Electric and
Fuel EUI statistics side by side.

## 1. Download the benchmark tables from BPD

BPD requires a free account. The downloads are **not provided with this project**.

1. Open **https://bpd.lbl.gov** and sign in.
2. In the filter panel on the right, open **Building Type**. Everything under *Commercial* is selected by default:
   **deselect all**, then select **Education**, and within Education select only the **Preschool/Daycare** and
   **K-12 School** options.
3. Open **Location and Data Source** → **Climate Zone**. All zones are selected by default: **deselect all**, then
   select **one** zone (e.g. `4A`).
4. Scroll down to the **Table** card:
   - **Group By:** `Building Type`
   - **Analyze By:** `Site EUI` (the default is Source EUI — change it)
5. Click **Download**. Rename the file to `<zone>_Site_EUI.csv` (e.g. `4A_Site_EUI.csv`).
6. With the same zone selected, change **Analyze By** to `Electric EUI` and download as `<zone>_Electric_EUI.csv`,
   then to `Fuel EUI` and download as `<zone>_Fuel_EUI.csv`. Both are required: site EUI alone does not say how a
   school's energy splits between electricity and fuel, and later steps need that split to estimate cost.
7. Repeat steps 3–6 for **every climate zone**.
8. Put all files in `data/raw/bpd/`.

```
data/raw/bpd/
├── 1A_Site_EUI.csv      1A_Electric_EUI.csv      1A_Fuel_EUI.csv
├── 2A_Site_EUI.csv      2A_Electric_EUI.csv      2A_Fuel_EUI.csv
└── …
```

A zone with no school buildings in BPD gives no usable rows; that zone simply has no benchmark (reported below).

BPD also offers an **API under a commercial license**, which would allow these tables — or building-level peer data —
to be pulled automatically in the future.

## 2. Setup

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)


def check(condition: bool, failure_message: str) -> None:
    # Stops the notebook with a plain-language message instead of continuing with bad data.
    if not condition:
        raise ValueError(failure_message)


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BPD_DOWNLOAD_DIR = PROJECT_ROOT / "data" / "raw" / "bpd"
OUTPUT_FILE = PROJECT_ROOT / "data" / "interim" / "bpd_eui_benchmarks.csv"

METRICS = ["Site", "Electric", "Fuel"]
FILE_NAME_PATTERN = re.compile(r"^(?P<climate_zone>[1-8][ABC]?)_(?P<metric>Site|Electric|Fuel)_EUI\.csv$")
# Column headers of a BPD "Group By: Building Type" table download → names used here.
BPD_COLUMNS = {
    "Facility Type": "building_type",
    "Count": "count",
    "Mean": "mean",
    "Standard Deviation": "std",
    "Min": "min",
    "25th Percentile": "p25",
    "50th Percentile": "p50",
    "75th Percentile": "p75",
    "Max": "max",
}
STATISTICS = ["count", "mean", "std", "min", "p25", "p50", "p75", "max"]


def climate_zone_sort_key(zone: str) -> tuple[int, str]:
    # Sorts zones in ASHRAE order: 1A, 2A, 2B, 3A, … 8 (number first, then moisture letter).
    return int(zone[0]), zone[1:]


check(
    BPD_DOWNLOAD_DIR.exists(),
    f"BPD download folder not found: {BPD_DOWNLOAD_DIR.relative_to(PROJECT_ROOT)}. Download the tables as described in section 1.",
)

## 3. Read every download

Each download lists every BPD building type; only the selected types have a count above zero. The notebook keeps
**every Education row with at least one building**, whatever BPD calls it, plus BPD's `Summary` row (all selected
buildings in that zone combined). BPD has renamed its school categories over time, so names are not hard-coded.

In [ ]:
download_files = sorted(BPD_DOWNLOAD_DIR.glob("*.csv"))
unexpected_names = [path.name for path in download_files if not FILE_NAME_PATTERN.match(path.name)]
check(download_files, f"No CSV files in {BPD_DOWNLOAD_DIR.relative_to(PROJECT_ROOT)}.")
check(
    not unexpected_names,
    f"Files not named <zone>_<Site|Electric|Fuel>_EUI.csv: {unexpected_names}. Rename them as in section 1.",
)

tables = []
for path in download_files:
    name_parts = FILE_NAME_PATTERN.match(path.name)
    # BPD downloads start with a byte-order mark; utf-8-sig removes it so the first header reads "Facility Type".
    table = pd.read_csv(path, encoding="utf-8-sig")
    missing_columns = set(BPD_COLUMNS) - set(table.columns)
    check(
        not missing_columns,
        f"{path.name} is missing columns {missing_columns}. Was it downloaded with Group By = Building Type?",
    )
    table = table.rename(columns=BPD_COLUMNS)[list(BPD_COLUMNS.values())]
    table["climate_zone"] = name_parts["climate_zone"]
    table["metric"] = name_parts["metric"]
    tables.append(table)
bpd_raw = pd.concat(tables, ignore_index=True)

# A building type outside Education with buildings means the Building Type filter was not set as in section 1.
other_types = bpd_raw[
    ~bpd_raw["building_type"].str.startswith("Education")
    & (bpd_raw["building_type"] != "Summary")
    & (bpd_raw["count"] > 0)
]
other_type_rows = other_types[["climate_zone", "metric", "building_type", "count"]]
check(
    other_types.empty,
    f"Non-education building types have data — check the Building Type filter:\n{other_type_rows}",
)

bpd_long = bpd_raw[
    (bpd_raw["building_type"].str.startswith("Education") | (bpd_raw["building_type"] == "Summary"))
    & (bpd_raw["count"] > 0)
].copy()
bpd_long["building_type"] = bpd_long["building_type"].str.replace("Education - ", "", regex=False)
check(
    not bpd_long.duplicated(["climate_zone", "metric", "building_type"]).any(),
    "A climate zone/metric was downloaded twice with different file names.",
)

building_types_found = sorted(bpd_long["building_type"].unique())
print(f"Files read: {len(download_files)} | climate zones: {bpd_long['climate_zone'].nunique()}")
print(f"Building types found: {building_types_found}")

In [ ]:
# Which metrics were downloaded for each zone — every zone should have all three.
files_per_zone = bpd_raw.drop_duplicates(["climate_zone", "metric"]).pivot_table(
    index="climate_zone", columns="metric", values="building_type", aggfunc="size", fill_value=0
)
files_per_zone = files_per_zone.reindex(columns=METRICS, fill_value=0).astype(bool)
incomplete_zones = files_per_zone[~files_per_zone.all(axis=1)]
print(
    "Zones missing a metric download:"
    if len(incomplete_zones)
    else "Every zone has Site, Electric and Fuel downloads."
)
incomplete_zones if len(incomplete_zones) else files_per_zone.T

## 4. Build the benchmark table

One row per climate zone × building type. Statistics for each metric sit side by side, e.g. `site_eui_p50`,
`electric_eui_p50`, `fuel_eui_p50`. The building count can differ slightly between metrics because not every
building reports both electricity and fuel.

In [ ]:
benchmarks = bpd_long.pivot_table(
    index=["climate_zone", "building_type"], columns="metric", values=STATISTICS, aggfunc="first"
)
benchmarks.columns = [f"{metric.lower()}_eui_{statistic}" for statistic, metric in benchmarks.columns]
ordered_columns = [f"{metric.lower()}_eui_{statistic}" for metric in METRICS for statistic in STATISTICS]
benchmarks = benchmarks.reindex(columns=ordered_columns).reset_index()

# Climate zones in ASHRAE order (1A, 2A, 2B, … 8), building types alphabetically with Summary last.
benchmarks["zone_order"] = benchmarks["climate_zone"].map(climate_zone_sort_key)
benchmarks["type_order"] = benchmarks["building_type"].eq("Summary")
benchmarks = (
    benchmarks.sort_values(["zone_order", "type_order", "building_type"])
    .drop(columns=["zone_order", "type_order"])
    .reset_index(drop=True)
)
for column in [c for c in benchmarks.columns if c.endswith("_count")]:
    benchmarks[column] = benchmarks[column].astype("Int64")

benchmarks[
    [
        "climate_zone",
        "building_type",
        "site_eui_count",
        "site_eui_p25",
        "site_eui_p50",
        "site_eui_p75",
        "electric_eui_p50",
        "fuel_eui_p50",
    ]
]

In [ ]:
# Sanity checks on the published statistics: percentiles in order, EUI within min–max.
for metric in METRICS:
    prefix = f"{metric.lower()}_eui_"
    rows = benchmarks.dropna(subset=[prefix + "p50"])
    ordered = (
        (rows[prefix + "min"] <= rows[prefix + "p25"])
        & (rows[prefix + "p25"] <= rows[prefix + "p50"])
        & (rows[prefix + "p50"] <= rows[prefix + "p75"])
        & (rows[prefix + "p75"] <= rows[prefix + "max"])
    )
    check(
        ordered.all(),
        f"{metric} EUI: percentiles out of order for\n{rows.loc[~ordered, ['climate_zone', 'building_type']]}",
    )
print("Percentiles are in order (min ≤ 25th ≤ 50th ≤ 75th ≤ max) for every zone, building type and metric.")

## 5. Call-outs: known limits of these benchmarks

1. **Self-selected sample.** BPD only contains buildings whose owners reported data. The schools in a zone are not a
   random sample of that zone's schools.
2. **Anonymized locations.** BPD does not say where in the climate zone its buildings are, so a benchmark cannot be
   narrowed to a state or region.
3. **Small samples.** Some zone × building type cells hold few buildings (see the counts below); their percentiles
   are unstable.
4. **Building types change.** BPD has regrouped its school categories over time — earlier downloads split
   *Elementary or middle school* from *High school*; current downloads group all grades under K-12. When only one
   K-12 category exists, elementary, middle and high schools all receive the same benchmark.
5. **Zones without data.** A climate zone with no school buildings in BPD has no benchmark (listed below).
6. **Site EUI, not source EUI.** Site EUI counts energy used at the building; it does not include losses in
   generating and delivering electricity.
7. **Use the *Other classroom* category cautiously.** It is BPD's broadest education category — in earlier downloads
   it held most of the buildings in a zone (5,760 of 6,922 in zone 4A) — and it does not identify the kind of school.
   Prefer the specific school categories where they have enough buildings.

**Note — model-based alternative.** A model trained on K-12 school data to estimate a school's site EUI from its
building characteristics was developed after the SEED prototype. It was not used in the SEED prototype or in these
benchmarks; it can be shown if useful.

In [ ]:
# Call-out 3: building counts behind each benchmark (Site EUI).
benchmarks.pivot_table(
    index="building_type", columns="climate_zone", values="site_eui_count", aggfunc="first"
)[sorted(benchmarks["climate_zone"].unique(), key=climate_zone_sort_key)]

In [ ]:
# Call-out 5: climate zones that schools are in (from Step 3, if it has been run) but BPD has no benchmark for.
print(
    f"Climate zones with a benchmark: {sorted(benchmarks['climate_zone'].unique(), key=climate_zone_sort_key)}"
)
school_zones_file = PROJECT_ROOT / "data" / "interim" / "school_climate_zone_2024_25.csv"
if school_zones_file.exists():
    school_zones = pd.read_csv(school_zones_file, dtype=str)["climate_zone"].dropna()
    uncovered = (
        school_zones[~school_zones.isin(benchmarks["climate_zone"])]
        .value_counts()
        .rename("operational schools")
    )
    print(
        "Every school climate zone has a benchmark."
        if uncovered.empty
        else "School climate zones with no benchmark:"
    )
    display(uncovered.to_frame()) if not uncovered.empty else None
else:
    print("Run Step 3 to list school climate zones that have no benchmark.")

## 6. Visual check

In [ ]:
site = benchmarks.dropna(subset=["site_eui_p50"])
zones = sorted(site["climate_zone"].unique(), key=climate_zone_sort_key)
# Building types alphabetically, with BPD's all-types "Summary" last.
building_types = sorted(site["building_type"].unique(), key=lambda name: (name == "Summary", name))
bar_width = 0.8 / len(building_types)

fig, ax = plt.subplots(figsize=(14, 5))
for position, building_type in enumerate(building_types):
    rows = site[site["building_type"] == building_type].set_index("climate_zone").reindex(zones)
    x = np.arange(len(zones)) + (position - (len(building_types) - 1) / 2) * bar_width
    ax.bar(x, rows["site_eui_p50"], width=bar_width, label=building_type)
    # Whiskers show the 25th–75th percentile range.
    ax.errorbar(
        x,
        rows["site_eui_p50"],
        yerr=[rows["site_eui_p50"] - rows["site_eui_p25"], rows["site_eui_p75"] - rows["site_eui_p50"]],
        fmt="none",
        ecolor="black",
        elinewidth=0.8,
        capsize=2,
    )
ax.set_xticks(np.arange(len(zones)), zones)
ax.set_title("Median site EUI by climate zone and building type (whiskers: 25th–75th percentile)")
ax.set_xlabel("ASHRAE climate zone")
ax.set_ylabel("Site EUI (kBtu/ft²·yr)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 7. Save output

`bpd_eui_benchmarks.csv` — one row per climate zone × building type:

| Column | Meaning |
|---|---|
| `climate_zone` | ASHRAE climate zone, from the file name |
| `building_type` | BPD building type (Education subtypes as BPD names them; `Summary` = all selected types combined) |
| `site_eui_count` … `site_eui_max` | Number of buildings, mean, standard deviation, min, 25th/50th/75th percentile and max of **site** EUI |
| `electric_eui_*`, `fuel_eui_*` | The same statistics for **electric** and **fuel** EUI |

EUI is in kBtu per square foot per year.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
benchmarks.to_csv(OUTPUT_FILE, index=False)
reread = pd.read_csv(OUTPUT_FILE)
check(len(reread) == len(benchmarks), "Saved benchmark file does not match what was computed.")
print(f"Saved {len(reread)} benchmark rows to {OUTPUT_FILE.relative_to(PROJECT_ROOT)}")